# Qwen3-8B 多层MLP TT-matrix 替换实验

In [1]:
import sys
from pathlib import Path
from pprint import pprint

import torch

PROJECT_ROOT = Path("/home/xls/workspace/projects/qwen3-tn-compression").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# 自动重新加载 src 下已修改的模块，调试源码后无需重启 kernel。
ipython = get_ipython()
ipython.extension_manager.load_extension("autoreload")
ipython.run_line_magic("autoreload", "2")

from qwen3_tn import TTMatrixExperiment, build_qwen_mlp_tt_matrix_targets

print("项目目录：", PROJECT_ROOT)
print("PyTorch：", torch.__version__)

项目目录： /mnt/intern7/xls/projects/qwen3-tn-compression
PyTorch： 2.5.1


## 1. 配置

同类 projection 共享 modes/ranks，个别层可通过 `LAYER_OVERRIDES` 覆盖。`prod(out_modes)` 和 `prod(in_modes)` 必须分别等于目标 `Linear` 的输出、输入维度。

In [2]:
MODEL_PATH = Path("/infini-data/Qwen3-8B")
LAYER_INDICES = list(range(0, 7)) # 全部 36 层改为 list(range(36))
# 6层还可以，再多就乱码了

PROJECTION_CONFIGS = {
    "down_proj": {
        # down_proj.weight: [4096, 12288]
        "out_modes": (8, 8, 8, 8),       # 乘积 4096
        "in_modes": (8, 8, 8, 24),       # 乘积 12288
        "ranks": (1, 64, 2048, 192, 1),  # 中央 rank 4096 -> 2048
        "token_chunk_size": 8,
    },
}
LAYER_OVERRIDES = {
    # (1, "down_proj"): {"ranks": (1, 64, 1024, 192, 1)},
}

SVD_DRIVER = "gesvd"
MIN_FREE_GPU_GIB = 20
CACHE_ROOT = PROJECT_ROOT / "artifacts/tt_matrix_cache"
FORCE_RECOMPUTE = False  # False：元数据和权重 SHA-256 一致时直接加载缓存
PROMPT = "请用三句话介绍上海，并说明最适合游览的季节。"
MAX_NEW_TOKENS = 256
BENCHMARK_NEW_TOKENS = 64
BENCHMARK_WARMUP_RUNS = 1
BENCHMARK_REPEATS = 3
RESTORE_AFTER_COMPARE = False  # False：成功后保留 TTMatrixLinear
SAVE_ROOT = PROJECT_ROOT / "artifacts/multi_layer_down_proj"

targets = build_qwen_mlp_tt_matrix_targets(
    LAYER_INDICES,
    PROJECTION_CONFIGS,
    LAYER_OVERRIDES,
)
print("目标数量：", len(targets))
for target in targets:
    print(target.module_path, target.ranks)

目标数量： 7
model.layers.0.mlp.down_proj (1, 64, 2048, 192, 1)
model.layers.1.mlp.down_proj (1, 64, 2048, 192, 1)
model.layers.2.mlp.down_proj (1, 64, 2048, 192, 1)
model.layers.3.mlp.down_proj (1, 64, 2048, 192, 1)
model.layers.4.mlp.down_proj (1, 64, 2048, 192, 1)
model.layers.5.mlp.down_proj (1, 64, 2048, 192, 1)
model.layers.6.mlp.down_proj (1, 64, 2048, 192, 1)


## 2. 加载或复用完整模型

同一 Jupyter kernel、同一实验对象重复调用 `load_model()` 不会重复读取权重。不同 Notebook kernel 是独立 Python 进程，无法共享模型；若 `nvidia-smi` 显示多个约 16 GiB 的进程，应关闭不再使用的 kernel。


In [3]:
session_key = (
    str(MODEL_PATH.resolve()),
    "cuda:0",
    str(torch.bfloat16),
    float(MIN_FREE_GPU_GIB),
)
if (
    "experiment" not in globals()
    or globals().get("_TT_MATRIX_SESSION_KEY") != session_key
):
    old_experiment = globals().get("experiment")
    if old_experiment is not None:
        old_experiment.close()
    experiment = TTMatrixExperiment(
        MODEL_PATH,
        device="cuda:0",
        model_dtype=torch.bfloat16,
        core_dtype=torch.bfloat16,
        min_free_gpu_gib=MIN_FREE_GPU_GIB,
    )
    _TT_MATRIX_SESSION_KEY = session_key

model = experiment.load_model()
print("模型已就绪；重复运行本格不会重复加载：", experiment.model_path)

/home/xls/appdata/miniforge3/envs/qwen3-tn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 5/5 [00:05<00:00,  1.13s/it]

模型已就绪；重复运行本格不会重复加载： /infini-data/Qwen3-8B


## 3. 逐层分解

每次只为一个目标创建 FP32 工作副本，得到的 BF16 TT-matrix cores 保存在 CPU。缓存会先匹配元数据，再验证原始权重 SHA-256；每层完成后立即保存，因此中断后可以续跑。再次运行会先自动恢复上一轮安装的稠密层。

In [4]:
decomposition = experiment.decompose(
    targets,
    svd_driver=SVD_DRIVER,
    cache_root=CACHE_ROOT,
    force_recompute=FORCE_RECOMPUTE,
    verbose=True,
)
pprint(decomposition["aggregate"])
for module_path, metrics in decomposition["targets"].items():
    print(
        module_path,
        f"compression={metrics['compression_ratio']:.3f}x",
        f"weight_relative_l2={metrics['weight']['relative_l2']:.6f}",
        "source=cache" if metrics['cache_hit'] else "source=decomposed",
        (
            f"cache_load={metrics['cache_load_seconds']:.2f}s"
            if metrics['cache_hit']
            else f"svd={metrics['svd_seconds']:.2f}s"
        ),
    )

if SAVE_ROOT is not None:
    experiment.save_checkpoints(SAVE_ROOT)
    print("TT-matrix checkpoints 已保存到：", SAVE_ROOT)

[1/7] model.layers.0.mlp.down_proj：检查缓存元数据
[1/7] model.layers.0.mlp.down_proj：元数据命中，验证权重 SHA-256


[1/7] model.layers.0.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[2/7] model.layers.1.mlp.down_proj：检查缓存元数据
[2/7] model.layers.1.mlp.down_proj：元数据命中，验证权重 SHA-256
[2/7] model.layers.1.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[3/7] model.layers.2.mlp.down_proj：检查缓存元数据
[3/7] model.layers.2.mlp.down_proj：元数据命中，验证权重 SHA-256
[3/7] model.layers.2.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[4/7] model.layers.3.mlp.down_proj：检查缓存元数据
[4/7] model.layers.3.mlp.down_proj：元数据命中，验证权重 SHA-256
[4/7] model.layers.3.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[5/7] model.layers.4.mlp.down_proj：检查缓存元数据
[5/7] model.layers.4.mlp.down_proj：元数据命中，验证权重 SHA-256
[5/7] model.layers.4.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[6/7] model.layers.5.mlp.down_proj：检查缓存元数据
[6/7] model.layers.5.mlp.down_proj：元数据命中，验证权重 SHA-256
[6/7] model.layers.5.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[7/7] model.layers.6.mlp.down_proj：检查缓存元数据
[7/7] model.layers.6.mlp.down_proj：元数据命中，验证权重 SHA-256
[7/7] model.layers.6.mlp.down_proj：缓存命中，

## 4. 批量换层并对比推理

默认成功后保留所有 `TTMatrixLinear`。如需推理结束后自动恢复稠密层，将 `RESTORE_AFTER_COMPARE` 设为 `True`；无论该设置如何，异常都会触发整体恢复。

In [5]:
comparison = experiment.compare(
    PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    benchmark_new_tokens=BENCHMARK_NEW_TOKENS,
    benchmark_warmup_runs=BENCHMARK_WARMUP_RUNS,
    benchmark_repeats=BENCHMARK_REPEATS,
    restore_after=RESTORE_AFTER_COMPARE,
)
print(experiment.format_comparison(comparison))

/home/xls/appdata/miniforge3/envs/qwen3-tn/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/xls/appdata/miniforge3/envs/qwen3-tn/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/xls/appdata/miniforge3/envs/qwen3-tn/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


完整模型压缩
  原始参数量：8,190,735,360（8.19B）
  替换后参数量：8,073,581,568（8.07B）
  减少参数量：117,153,792（117.15M）
  参数减少比例：1.43%
  参数压缩率：1.015x（压缩）

目标矩阵合计
  矩阵数量：7
  原始参数量：352,321,536（352.32M）
  TT-matrix 参数量：235,167,744（235.17M）
  减少参数量：117,153,792（117.15M）
  参数减少比例：33.25%
  参数压缩率：1.498x（压缩）

总体矩阵重构误差
  relative L2：5.085e-01（50.85%）

单次推理资源开销
  稠密单次总耗时：2.49 秒
  TT-matrix 单次总耗时：7.77 秒
  稠密峰值显存：15.29 GiB
  TT-matrix 峰值显存：15.31 GiB

推理速度基准
  设置：预热 1 次，测量 3 次，每次固定生成 64 token
  稠密生成速度：35.72 token/s
  TT-matrix 生成速度：32.77 token/s
  稠密中位耗时：1.79 秒
  TT-matrix 中位耗时：1.95 秒
  相对速度：0.918x（慢 8.24%）

稠密模型回答：
上海是中国最具现代化和国际化的大都市之一，拥有繁华的陆家嘴金融区、历史悠久的外滩和丰富的文化景点。作为一座融合中西文化的城市，它以独特的魅力吸引着世界各地的游客。最适合游览的季节是春秋，气候宜人，景色优美，既能避开夏季的酷暑和冬季的湿冷，又能充分体验城市的魅力。

TT-matrix 模型回答：
上海是适合游览的季节。最适合游览的季节是上海。上海是适合游览的季节。最适合游览的季节是上海。上海是适合游览的季节。最适合游览的季节是上海。上海是适合游览的季节。最适合游览的季节是上海。上海是适合游览的季节。最适合游览的季节是上海。上海是适合游览的季节。最适合游览的季节是上海。上海是适合游览的季节。最适合游览的季节是上海。上海是适合游览的季节。最适合游览的季节是上海。上海是适合游览的季节。最适合游览的季节是上海。上海是适合游览的季节。最适合游览的季节是上海。上海是适合游览的季节。最适合游览的季节是上海。上海是适合游览的季节。最适